In [ ]:
"""
UMAP visualization of pretrained ResNet-50 feature vectors on CIFAR-10.

CIFAR-10 classes (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck)
are all present in ImageNet-1k, so this visualizes the model on its own
training categories — expect clean, well-separated clusters.

Requirements:
    pip install transformers datasets umap-learn plotly tqdm torch pillow
"""

import numpy as np
import torch
from transformers import AutoImageProcessor, AutoModel
from datasets import load_dataset
from umap import UMAP
import plotly.express as px
import pandas as pd
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
MODEL_ID          = "microsoft/resnet-50"
DATASET_ID        = "uoft-cs/cifar10"
SPLIT             = "test"       # 10k images, faster than 50k train
BATCH_SIZE        = 32
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------------------------------------------------------
# Load model — AutoModel (NOT AutoModelForImageClassification)
# Gives backbone only; outputs.pooler_output is the 2048-dim feature vector
# ---------------------------------------------------------------------------
print(f"Loading model: {MODEL_ID}")
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model     = AutoModel.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

# ---------------------------------------------------------------------------
# Load dataset
# cifar10 columns: "img" (PIL Image), "label" (ClassLabel 0-9)
# ---------------------------------------------------------------------------
print(f"Loading dataset: {DATASET_ID} / {SPLIT}")
dataset     = load_dataset(DATASET_ID, split=SPLIT)
class_names = dataset.features["label"].names
print(f"Classes : {class_names}")
print(f"Images  : {len(dataset)}")

# ---------------------------------------------------------------------------
# Extract feature vectors
# ---------------------------------------------------------------------------
all_features = []
all_labels   = []

print("Extracting features...")
for start in tqdm(range(0, len(dataset), BATCH_SIZE)):
    batch  = dataset[start : start + BATCH_SIZE]

    # cifar10 column is "img"; ensure RGB
    images = [img.convert("RGB") for img in batch["img"]]
    labels = batch["label"]

    inputs = processor(images=images, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

    # pooler_output: (batch, 2048, 1, 1) → squeeze to (batch, 2048)
    features = outputs.pooler_output.squeeze(-1).squeeze(-1)
    all_features.append(features.cpu().numpy())
    all_labels.extend(labels)

features_array = np.concatenate(all_features, axis=0)   # (N, 2048)
labels_array   = np.array(all_labels)                   # (N,)
label_names    = [class_names[l] for l in labels_array]

print(f"Feature matrix: {features_array.shape}")

/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: microsoft/resnet-50


Loading weights: 100%|██████████| 318/318 [00:00<00:00, 1834.94it/s]
[transformers] ResNetModel LOAD REPORT from: microsoft/resnet-50
Key                 | Status     |  | 
--------------------+------------+--+-
classifier.1.bias   | UNEXPECTED |  | 
classifier.1.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading dataset: uoft-cs/cifar10 / test
Classes : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Images  : 10000
Extracting features...


100%|██████████| 313/313 [15:25<00:00,  2.96s/it]
/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Feature matrix: (10000, 2048)
Running UMAP...


Running UMAP...


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [5]:
# Keep these fixed when comparing models later
UMAP_N_NEIGHBORS  = 5
UMAP_MIN_DIST     = 0.01
UMAP_RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# UMAP
# ---------------------------------------------------------------------------
print("Running UMAP...")
reducer   = UMAP(
    n_neighbors   = UMAP_N_NEIGHBORS,
    min_dist      = UMAP_MIN_DIST,
    random_state  = UMAP_RANDOM_STATE,
    n_components  = 2,
)
embedding = reducer.fit_transform(features_array)   # (N, 2)

# ---------------------------------------------------------------------------
# Visualize — interactive HTML scatter
# ---------------------------------------------------------------------------
df = pd.DataFrame({
    "UMAP 1" : embedding[:, 0],
    "UMAP 2" : embedding[:, 1],
    "class"  : label_names,
})

fig = px.scatter(
    df,
    x       = "UMAP 1",
    y       = "UMAP 2",
    color   = "class",
    title   = f"UMAP of {MODEL_ID} features — {DATASET_ID} ({SPLIT})",
    width   = 900,
    height  = 700,
    opacity = 0.7,
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(legend_title_text="Class")

output_path = "umap_resnet50_cifar10.html"
fig.write_html(output_path)
print(f"Saved: {output_path}")
fig.show()

Running UMAP...


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved: umap_resnet50_cifar10.html
